# Importing libraries

In [9]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

from datasets import load_dataset

from sklearn.metrics import precision_score, recall_score, f1_score
from memory_profiler import memory_usage
from time import perf_counter

# Importing dataset

In [10]:
ds = load_dataset("christinacdl/binary_hate_speech")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df['label'] = train_df['label'].apply(lambda x: 1 if x == 'OFF_HATEFUL_TOXIC' else 0)
val_df['label'] = val_df['label'].apply(lambda x: 1 if x == 'OFF_HATEFUL_TOXIC' else 0)
test_df['label'] = test_df['label'].apply(lambda x: 1 if x == 'OFF_HATEFUL_TOXIC' else 0)

train_df

,text,label
0,She won't be there for long.,0
1,i guess eu is gonna have to back track a littl...,0
2,@user @user @user @user @user I can understand...,1
3,Media Matters hates Joe diGenova - that's a re...,0
4,@user @user @user @user thanks to the best b'd...,0
...,...,...
31055,Actual animals however do object 😏,0
31056,&#8220;@PubesOnFleeK: My tweets trash&#8221;,1
31057,Confusing circumstances seem to get in the way...,0
31058,now new reading material for my #entrepreneuri...,0


# Dataset preprocessing

In [11]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,  # Make sure this is False
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [12]:
MAX_LEN = 128
BATCH_SIZE = 32

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Create datasets
train_dataset = TextDataset(
    texts=train_df['text'],
    labels=train_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df['text'],
    labels=val_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df['text'],
    labels=test_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Neural network class (LSTM)

In [13]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(LSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        # 1. Embedding lookup
        embedded = self.embedding(input_ids)

        # 2. LSTM forward pass
        outputs, (hidden, cell) = self.lstm(embedded)

        # 3. Extract the final hidden state
        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        # 4. Apply dropout
        hidden = self.dropout(hidden)  

        # 5. Final classification layer (returns logits)
        output = self.fc(hidden)

        return output

# Instancing the LSTM model, criterion and optimizer

In [14]:
embedding_dim = 128
hidden_dim = 128
output_dim = 1
n_layers = 2
bidirectional = True
dropout = 0.3

model = LSTMClassifier(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')
model = model.to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Using cuda device


# Training and evaluation functions

In [16]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    """
    One epoch of training. 
    - model: your LSTMClassifier (returns raw logits).
    - data_loader: iterable with 'input_ids' and 'labels' in each batch.
    - optimizer, criterion: training components (e.g., Adam, BCEWithLogitsLoss).
    - device: 'cpu' or 'cuda'.
    """
    model.train()
    losses = []
    correct_predictions = 0

    # For calculating precision, recall, F1:
    all_labels = []
    all_preds = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # 1) Forward pass -> raw logits
        logits = model(input_ids)  # shape: (batch_size, 1)
        logits = logits.squeeze(dim=1)  # shape: (batch_size,)

        # 2) Compute loss (BCEWithLogitsLoss expects raw logits)
        loss = criterion(logits, labels.float())

        # 3) Backprop + optimization
        loss.backward()
        optimizer.step()

        # 4) Track loss
        losses.append(loss.item())

        # 5) Convert logits -> probabilities -> predicted classes
        probs = torch.sigmoid(logits)          # in [0, 1]
        preds_cls = (probs >= 0.5).long()      # threshold at 0.5

        # 6) Count correct predictions
        correct_predictions += torch.sum(preds_cls == labels)

        # 7) Collect for metric calculation
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds_cls.cpu().numpy())

    # Calculate overall metrics for the epoch
    precision = precision_score(all_labels, all_preds, zero_division=0,average='macro')
    recall = recall_score(all_labels, all_preds, zero_division=0,average='macro')
    f1 = f1_score(all_labels, all_preds, zero_division=0,average='macro')
    accuracy = float(correct_predictions) / len(data_loader.dataset)
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss, precision, recall, f1


def eval_model(model, data_loader, criterion, device):
    """
    Evaluation function. Similar to train_epoch, but no backprop.
    - model: your LSTMClassifier (returns raw logits).
    - data_loader: iterable with 'input_ids' and 'labels'.
    - criterion: e.g., BCEWithLogitsLoss for binary classification.
    - device: 'cpu' or 'cuda'.
    """
    model.eval()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            # 1) Forward pass -> logits
            logits = model(input_ids)  # shape: (batch_size, 1)
            logits = logits.squeeze(dim=1)  # shape: (batch_size,)

            # 2) Compute loss
            loss = criterion(logits, labels.float())
            losses.append(loss.item())

            # 3) Convert logits -> probabilities -> predicted classes
            probs = torch.sigmoid(logits)
            preds_cls = (probs >= 0.5).long()

            correct_predictions += torch.sum(preds_cls == labels)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds_cls.cpu().numpy())

    # Metrics
    precision = precision_score(all_labels, all_preds, zero_division=0, average='macro')
    recall = recall_score(all_labels, all_preds, zero_division=0, average='macro')
    f1 = f1_score(all_labels, all_preds, zero_division=0, average='macro')
    accuracy = float(correct_predictions) / len(data_loader.dataset)
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss, precision, recall, f1

# Training loop

In [17]:
def training_loop(epochs):
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        
        train_acc, train_loss, train_prec, train_rec, train_f1 = train_epoch(
            model, train_loader, optimizer, criterion, device)
        
        val_acc, val_loss, val_prec, val_rec, val_f1 = eval_model(
            model, val_loader, criterion, device)
        
        print(f'Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, '
            f'Precision: {train_prec:.4f}, Recall: {train_rec:.4f}, F1 Score: {train_f1:.4f}')
        
        print(f'Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, '
            f'Precision: {val_prec:.4f}, Recall: {val_rec:.4f}, F1 Score: {val_f1:.4f}')
    return train_acc, train_loss, train_prec, train_rec, train_f1, val_acc, val_loss, val_prec, val_rec, val_f1

In [18]:
seeds = [2,3,5]
EPOCHS = 5
results = pd.DataFrame(columns=['seed', 'train_loss', 'train_acc', 'train_prec', 'train_rec', 'train_f1',
                                'val_loss', 'val_acc', 'val_prec', 'val_rec', 'val_f1',
                                'test_loss', 'test_acc', 'test_prec', 'test_rec', 'test_f1',
                                'max_memory_usage_train', 'max_vram_usage_train', 'total_time_train',
                                'max_memory_usage_test', 'max_vram_usage_test', 'total_time_test'])

for seed in seeds:
    torch.manual_seed(seed)
    # Resetting model
    model = LSTMClassifier(
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        n_layers=n_layers,
        bidirectional=bidirectional,
        dropout=dropout
    )
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss().to(device)
    
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_train = perf_counter()
    max_memory_usage_train, retval = memory_usage((training_loop, (EPOCHS,), {}), retval=True, max_usage=True)
    total_time_train = perf_counter() - start_time_train

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    train_acc, train_loss, train_prec, train_rec, train_f1, val_acc, val_loss, val_prec, val_rec, val_f1 = retval

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_test = perf_counter()
    max_memory_usage_test, retval = memory_usage((eval_model, (model, test_loader, criterion, device), {}), retval=True, max_usage=True)
    total_time_test = perf_counter() - start_time_test

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    test_acc, test_loss, test_prec, test_rec, test_f1 = retval

    results = pd.concat([results, pd.DataFrame([[seed, train_loss, train_acc, train_prec, train_rec, train_f1,
                                                val_loss, val_acc, val_prec, val_rec, val_f1,
                                                test_loss, test_acc, test_prec, test_rec, test_f1,
                                                max_memory_usage_train, max_vram_usage_train, total_time_train,
                                                max_memory_usage_test, max_vram_usage_test, total_time_test]],
                                                columns=results.columns)], ignore_index=True)
    

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6074, Accuracy: 0.6724, Precision: 0.6747, Recall: 0.6724, F1 Score: 0.6713
Val Loss: 0.5933, Accuracy: 0.6922, Precision: 0.7056, Recall: 0.6923, F1 Score: 0.6872
Epoch 2/5
Train Loss: 0.5033, Accuracy: 0.7561, Precision: 0.7586, Recall: 0.7561, F1 Score: 0.7555
Val Loss: 0.5039, Accuracy: 0.7566, Precision: 0.7572, Recall: 0.7566, F1 Score: 0.7565
Epoch 3/5
Train Loss: 0.4286, Accuracy: 0.8123, Precision: 0.8140, Recall: 0.8123, F1 Score: 0.8120
Val Loss: 0.4949, Accuracy: 0.7744, Precision: 0.7808, Recall: 0.7744, F1 Score: 0.7731
Epoch 4/5
Train Loss: 0.3642, Accuracy: 0.8506, Precision: 0.8524, Recall: 0.8506, F1 Score: 0.8504
Val Loss: 0.4933, Accuracy: 0.7881, Precision: 0.7887, Recall: 0.7881, F1 Score: 0.7879
Epoch 5/5
Train Loss: 0.2996, Accuracy: 0.8830, Precision: 0.8848, Recall: 0.8830, F1 Score: 0.8829
Val Loss: 0.5066, Accuracy: 0.7924, Precision: 0.7959, Recall: 0.7924, F1 Score: 0.7918


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
C:\Users\Rafael\AppData\Local\Temp\ipykernel_24956\3438238464.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([[seed, train_loss, train_acc, train_prec, train_rec, train_f1,
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6065, Accuracy: 0.6690, Precision: 0.6724, Recall: 0.6690, F1 Score: 0.6673
Val Loss: 0.5644, Accuracy: 0.7195, Precision: 0.7228, Recall: 0.7196, F1 Score: 0.7185
Epoch 2/5
Train Loss: 0.4968, Accuracy: 0.7626, Precision: 0.7650, Recall: 0.7626, F1 Score: 0.7620
Val Loss: 0.5082, Accuracy: 0.7476, Precision: 0.7478, Recall: 0.7476, F1 Score: 0.7476
Epoch 3/5
Train Loss: 0.4163, Accuracy: 0.8167, Precision: 0.8186, Recall: 0.8167, F1 Score: 0.8164
Val Loss: 0.4931, Accuracy: 0.7698, Precision: 0.7718, Recall: 0.7698, F1 Score: 0.7693
Epoch 4/5
Train Loss: 0.3448, Accuracy: 0.8597, Precision: 0.8623, Recall: 0.8597, F1 Score: 0.8595
Val Loss: 0.5250, Accuracy: 0.7628, Precision: 0.7656, Recall: 0.7628, F1 Score: 0.7622
Epoch 5/5
Train Loss: 0.2756, Accuracy: 0.8949, Precision: 0.8972, Recall: 0.8949, F1 Score: 0.8948
Val Loss: 0.5636, Accuracy: 0.7777, Precision: 0.7804, Recall: 0.7778, F1 Score: 0.7772


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6200, Accuracy: 0.6573, Precision: 0.6600, Recall: 0.6573, F1 Score: 0.6559
Val Loss: 0.5843, Accuracy: 0.7028, Precision: 0.7137, Recall: 0.7028, F1 Score: 0.6990
Epoch 2/5
Train Loss: 0.5215, Accuracy: 0.7464, Precision: 0.7499, Recall: 0.7464, F1 Score: 0.7455
Val Loss: 0.5178, Accuracy: 0.7543, Precision: 0.7565, Recall: 0.7543, F1 Score: 0.7538
Epoch 3/5
Train Loss: 0.4445, Accuracy: 0.7985, Precision: 0.8003, Recall: 0.7985, F1 Score: 0.7982
Val Loss: 0.5107, Accuracy: 0.7602, Precision: 0.7611, Recall: 0.7602, F1 Score: 0.7600
Epoch 4/5
Train Loss: 0.3844, Accuracy: 0.8367, Precision: 0.8384, Recall: 0.8367, F1 Score: 0.8365
Val Loss: 0.5111, Accuracy: 0.7677, Precision: 0.7752, Recall: 0.7677, F1 Score: 0.7661
Epoch 5/5
Train Loss: 0.3200, Accuracy: 0.8723, Precision: 0.8740, Recall: 0.8723, F1 Score: 0.8722
Val Loss: 0.5197, Accuracy: 0.7767, Precision: 0.7773, Recall: 0.7767, F1 Score: 0.7766


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [19]:
results.to_csv('results/lstm_binary1.csv', index=False)
results.head()

,seed,train_loss,train_acc,train_prec,train_rec,train_f1,val_loss,val_acc,val_prec,val_rec,...,test_acc,test_prec,test_rec,test_f1,max_memory_usage_train,max_vram_usage_train,total_time_train,max_memory_usage_test,max_vram_usage_test,total_time_test
0,2,0.299573,0.883033,0.884768,0.883033,0.882901,0.506607,0.792429,0.795888,0.792442,...,0.782900,0.786684,0.782885,0.782175,1346.894531,230.532227,64.510356,1344.066406,193.575195,1.703627
1,3,0.275579,0.894945,0.897173,0.894945,0.894798,0.563622,0.777749,0.780405,0.777762,...,0.775431,0.777875,0.775419,0.774931,1362.171875,231.047852,73.682359,1362.156250,193.575195,1.724250
2,5,0.320004,0.872344,0.873979,0.872344,0.872204,0.519729,0.776719,0.777275,0.776725,...,0.780840,0.782835,0.780829,0.780448,1347.320312,231.389648,90.584873,1347.296875,194.239258,1.723328


In [20]:
torch.save(model.state_dict(), 'results/lstm_binary1.pth')